# Silver Layer – Medications Dataset

## Description
This notebook cleans and standardizes the medications dataset from the Bronze layer.

The Silver layer performs basic data quality improvements and prepares the dataset for
downstream joins with diseases, inventory, and insurance tables.

## Source
capstone.bronze.medications

## Target
capstone.silver.medications

## Transformations
- Trim whitespace from identifiers and text fields
- Normalize medication and manufacturer names
- Standardize disease reference ID
- Normalize route values
- Normalize Rx/OTC flag
- Safely parse expiry dates

In [0]:
%python
from pyspark.sql.functions import *

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [0]:
%python
from pyspark.sql.functions import *

spark.sql("USE CATALOG capstone")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df = spark.table("capstone.bronze.medications")

silver = (
    df
    # --------------------------------------------------
    # Clean keys
    # --------------------------------------------------
    .withColumn("medication_id", upper(trim(col("medication_id"))))

    # --------------------------------------------------
    # Normalize core descriptive fields
    # --------------------------------------------------
    .withColumn("medication_name", initcap(trim(col("medication_name"))))
    .withColumn("active_ingredient", lower(trim(col("active_ingredient"))))
    .withColumn("dosage_form", lower(trim(col("dosage_form"))))
    .withColumn("manufacturer", initcap(trim(col("manufacturer"))))

    # --------------------------------------------------
    # Normalize route_raw -> route
    # --------------------------------------------------
    .withColumn(
        "route",
        when(lower(trim(col("route_raw"))).isin("po", "oral", "by mouth"), "oral")
        .when(lower(trim(col("route_raw"))).isin("iv", "intravenous"), "iv")
        .when(lower(trim(col("route_raw"))).isin("sc", "subcutaneous"), "subcutaneous")
        .when(lower(trim(col("route_raw"))).isin("top", "topical"), "topical")
        .when(lower(trim(col("route_raw"))).isin("inh", "inhalation", "nebulized"), "inhalation")
        .when(lower(trim(col("route_raw"))).isin("intranasal", "nasal"), "nasal")
        .otherwise(lower(trim(col("route_raw"))))
    )

    # --------------------------------------------------
    # Clean associated_disease_id_source -> disease_id
    # --------------------------------------------------
    .withColumn(
        "disease_id",
        upper(
            trim(
                regexp_replace(col("associated_disease_id_source"), "-OLD", "")
            )
        )
    )

    # --------------------------------------------------
    # Normalize rx_otc_flag_raw -> rx_otc_flag
    # --------------------------------------------------
    .withColumn(
        "rx_otc_flag",
        when(lower(trim(col("rx_otc_flag_raw"))).isin("rx", "prescription", "legend drug"), "RX")
        .when(lower(trim(col("rx_otc_flag_raw"))).isin("otc", "over the counter"), "OTC")
        .otherwise(None)
    )

    # --------------------------------------------------
    # expiry_date_raw -> expiry_date
    # --------------------------------------------------
    .withColumn(
        "expiry_date",
        coalesce(
            expr("try_to_date(expiry_date_raw, 'yyyy-MM-dd')"),
            expr("try_to_date(expiry_date_raw, 'MM/dd/yyyy')"),
            expr("try_to_date(expiry_date_raw, 'dd-MMM-yyyy')"),
            expr("try_to_date(expiry_date_raw, 'yyyy/MM/dd')")
        )
    )

    # --------------------------------------------------
    # Rename raw text columns
    # --------------------------------------------------
    .withColumn("warning_notes", trim(col("warning_notes_raw")))
    .withColumn("contraindications", trim(col("contraindications_raw")))
)

final_df = silver.select(
    "medication_id",
    "medication_name",
    "active_ingredient",
    "dosage_form",
    "manufacturer",
    "disease_id",
    "route",
    "rx_otc_flag",
    "expiry_date",
    "short_description",
    "warning_notes",
    "contraindications",
    "source_system",
    "source_record_id"
)

spark.sql("DROP TABLE IF EXISTS capstone.silver.medications")

(
    final_df.write
    .mode("overwrite")
    .format("delta")
    .saveAsTable("capstone.silver.medications")
)

print("Loaded: capstone.silver.medications")
print(final_df.dtypes)